# Notebook For exploration

In [1]:
import ee 
import epistemx
import geemap
#Option 1: Manual authenticate using personal account
#Instructions for manual authentication
epistemx.print_auth_instructions()
#uncomment the below line and follow earth engine authentication process
#epistemx.authenticate_manually()

#Option 2: Autheticate using service account (json file)
service_account_path = "C:/Users/AFahrezi/Documents/GitHub/EpistemXBackend/auth/earth-engine-451407-520c1ef64879.json"
#service_account_path = "C:/Users/AFahrezi/Documents/GitHub/ee-agilakbarfahrezi-023b11c810d5.json"
success = epistemx.initialize_with_service_account(service_account_path)

if success:
    print("Earth Engine initialized with service account successfully!")
else:
    print("Service account initialization failed. Try to authenticate earth engine manually")

#Check authentication status
status = epistemx.get_auth_status()
print(f"Initialized: {status['initialized']}")
print(f"Authenticated: {status['authenticated']}")
if status['project']:
    print(f"Project: {status['project']}")


    EARTH ENGINE AUTHENTICATION NOTES:
    
    1. Make sure you already have a google cloud project that has enable the Earth Engine API and registered to 
       commercial or non-commercial use. For more information visit: https://developers.google.com/earth-engine/guides/access 
    
    2. you can authenticate programmatically by calling: from epistemx.ee_config import authenticate_manually
       authenticate_manually()
    
    3. This will open a web browser. Sign in with your Google account that has Earth Engine access.
    
    4. Copy the authorization code from the browser and paste it in the terminal.
    
    
    For more details, visit: https://developers.google.com/earth-engine/guides/python_install
    
Earth Engine initialized with service account successfully!
Initialized: True
Authenticated: True


In [2]:
#This code is used if the notebook is implemented in github codespace. Just remove the (#)
!python -m pip install .. --quiet


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\AFahrezi\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Defining Area of Interest 
Use indonesia regency shapefile. Filter based on regency name or province name.

In [3]:
from epistemx.helpers import get_aoi_from_gaul
#FAO GAUL
#aoi = get_aoi_from_gaul(country="Indonesia", province="Sumatera Selatan")
#Indonesian National Border area.
#indo_regency = geemap.shp_to_ee('D:/EPISTEM-Agil/Spatial-Data/Indonesian_Regency_Area.shp')
indo_regency = ee.FeatureCollection('projects/ee-agilakbar/assets/Indonesian_Regency')
#check Regency List (required significant computation time)
regency = indo_regency.aggregate_array("WADMKK").getInfo()
province = indo_regency.aggregate_array("WADMPR").getInfo()
#Pagar Alam
regency_name = "Kota Pagar Alam"
# Filter the FeatureCollection
single_aoi = indo_regency.filter(ee.Filter.eq("WADMKK", regency_name))
#Area of Interest Geometry
aoi = single_aoi.geometry()
#garut selatan('D:/EPISTEM-Agil/Testing data/AOI_GARSEL/Extent_AOI_Proj.shp')
start = '2024-01-01'
end = '2024-12-31'

In [ ]:
#Get the AOI for Pagar Alam Regency
aoi = geemap.shp_to_ee('D:/EPISTEM-Agil/Spatial-Data/Pagar_Alam.shp')
start = '2024-01-01'
end = '2024-12-31'

## Retrive Landsat data from google earth engine

In [4]:
import geemap
#manual retrival
initial_collection = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
                            .filterBounds(aoi)\
                            .filterDate(start, end))\
                            .filterMetadata('CLOUD_COVER', 'less_than', 30)

# Applies scaling factors.
def apply_scale_factors(image):
  optical_bands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
  thermal_bands = image.select('ST_B.*').multiply(0.00341802).add(149.0)
  return image.addBands(optical_bands, None, True).addBands(
      thermal_bands, None, True
  )

dataset = initial_collection.map(apply_scale_factors)

rename_collection = dataset.select(
                            ['SR_B1','SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'], 
                            ['AEROSOL','BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2'])
mosaic = rename_collection.qualityMosaic('NIR').clip(aoi)
median = rename_collection.median().clip(aoi)
l8_sr_visparam = {'min': 0,'max': 0.4,'gamma': [0.95, 1.1, 1],'bands':['NIR', 'RED', 'GREEN']}
Map = geemap.Map()
Map.centerObject(aoi, 7)
Map.addLayer(rename_collection, l8_sr_visparam, "Initial Collection")
Map.addLayer(mosaic, l8_sr_visparam, "Mosaic Image")
Map.addLayer(median, l8_sr_visparam, "Median Image")
Map

Map(center=[-4.1175978213298485, 103.26638606871309], controls=(WidgetControl(options=['position', 'transparen…

In [5]:
from epistemx.module_1 import Reflectance_Data, Reflectance_Stats
optical_reflectance = Reflectance_Data()
#get the image collection and corresponding statistics
landsat_aoi_cloud, meta = optical_reflectance.get_multispectral_data(aoi=aoi, start_date=start, end_date=end, optical_data='L8_SR', 
                                                           cloud_cover=40, aoi_cloud_cover=20, aoi_cloud_scale=30, compute_detailed_stats=False)
landsat_data, stat = optical_reflectance.get_multispectral_data(aoi=aoi, start_date=start, end_date=end, optical_data='L8_SR', 
                                                           cloud_cover=40, compute_detailed_stats=False)

thermal_data, stats = optical_reflectance.get_thermal_bands(aoi=aoi, start_date=start, end_date=end,thermal_data='L8_TOA', 
                                                           cloud_cover=50, compute_detailed_stats=False)
# Define proper visualization parameters
vis_params = {
    'bands': ['NIR', 'RED', 'GREEN'],  # False color
    'min': 0.0,
    'max': 0.3,
    'gamma': [0.95, 1.1, 1]
}
Map = geemap.Map()
Map.centerObject(aoi, 7)
Map.addLayer(landsat_aoi_cloud, vis_params, "Landsat with aoi cloud")
Map.addLayer(landsat_data, vis_params, "Landsat with scene cloud")
Map.addLayer(aoi, {}, "single_aoi")
Map                                                   

2025-12-03 10:16:46,620 - Reflectance_Data - INFO - ReflectanceData initialized.
2025-12-03 10:16:46,621 - Reflectance_Data - INFO - Starting data fetch for Landsat 8 Operational Land Imager Surface Reflectance
2025-12-03 10:16:46,622 - Reflectance_Data - INFO - Date range: 2024-01-01 to 2024-12-31
2025-12-03 10:16:46,622 - Reflectance_Data - INFO - Cloud cover threshold: 40%
2025-12-03 10:16:46,623 - Reflectance_Data - INFO - AOI cloud cover threshold: 20% (scale: 30m)
2025-12-03 10:16:46,624 - Reflectance_Data - INFO - detailed statistics will not be computed
2025-12-03 10:16:46,625 - Reflectance_Stats - INFO - Reflectance Stats initialized.
2025-12-03 10:16:46,626 - Reflectance_Data - INFO - Filtered collection created (use compute_detailed_stats=True for more information)
2025-12-03 10:16:46,626 - Reflectance_Data - INFO - Computing cloud cover within AOI (this may take time for large areas)...
2025-12-03 10:16:46,638 - Reflectance_Data - INFO - Starting data fetch for Landsat 8 Op

Map(center=[-4.117597821329865, 103.26638606871309], controls=(WidgetControl(options=['position', 'transparent…

In [6]:
#intialize the statistic class
stats = Reflectance_Stats()
#get the retrival report and automatically print them
retrival_report = stats.get_collection_statistics(landsat_data, print_report=True)

2025-12-03 10:16:57,115 - Reflectance_Stats - INFO - Reflectance Stats initialized.


           Landsat Data Collection Retrival Report
Total Images Found: 6
Date Range: 2024-04-06 to 2024-10-15
Unique WRS Tiles: 1

Scene Cloud Cover Statistics:
------------------------------
Average Cloud Cover: 21.6%
Minimum Cloud Cover: 5.3%
Maximum Cloud Cover: 32.2%

WRS Path/Row Tiles:
------------------------------
Path 125/Row 063

Available Acqusition Date:
------------------------------
2024-04: 06, 22
2024-06: 09
2024-07: 27
2024-09: 13
2024-10: 15

Scene IDs (first 10):
------------------------------
• LC08_125063_20240406
• LC08_125063_20240422
• LC08_125063_20240609
• LC08_125063_20240727
• LC08_125063_20240913
• LC08_125063_20241015



In [21]:
from epistemx.module_1 import final_Image
final_data = final_Image()
mosaic = final_data.get_quality_mosaic(landsat_data, aoi, quality_band='NDVI')
temporal_composite = final_data.get_temporal_composite(landsat_data, aoi, reducer='Median')
#tmeporal_comp_cloud = final_data.get_temporal_composite(landsat_aoi_cloud, aoi, reducer='Median')
# Check what bands are available
print("Available bands:", mosaic.bandNames().getInfo())

# Define proper visualization parameters
false_color_ir = {
    'bands': ['NIR', 'RED', 'GREEN'],  # False color
    'min': 0.0,
    'max': 0.4,
    'gamma': [0.5, 0.9, 1]
}
true_color = {
    'bands': ['RED', 'GREEN', 'BLUE'],  # True color
    'min': 0.0,
    'max': 0.3,
    'gamma': [0.95, 1.1, 1]
}
# Create map and add layer WITH vis params
m = geemap.Map()
m.centerObject(aoi, 7)
m.addLayer(mosaic, false_color_ir, 'Mosaic')
m.addLayer(temporal_composite, false_color_ir, 'Temporal Composite')
m.addLayer(temporal_composite, true_color, 'Temporal Composite (True Color)')
#m.addLayer(tmeporal_comp_cloud, vis_params, "Temporal Composite with AOI cloud")
m.addLayer(aoi, {}, 'AOI Boundary', False)
m


2025-12-03 11:33:07,604 - final_Image - INFO - final_Image initialized.
2025-12-03 11:33:08,378 - final_Image - INFO - Creating quality mosaic from 6 images using NDVI as quality metric
2025-12-03 11:33:08,380 - final_Image - INFO - Quality mosaic created covering AOI with best available pixels
2025-12-03 11:33:09,433 - final_Image - INFO - Mosaic date range: 2024-04-06 to 2024-10-15
2025-12-03 11:33:10,079 - final_Image - INFO - Creating Median composite from 6 images
2025-12-03 11:33:10,080 - final_Image - INFO - Composite clipped to AOI
2025-12-03 11:33:11,695 - final_Image - INFO - Composite created from 2024-04-06 to 2024-10-15


Available bands: ['AEROSOL', 'BLUE', 'GREEN', 'RED', 'NIR', 'SWIR1', 'SWIR2']


Map(center=[-4.117597821329865, 103.26638606871309], controls=(WidgetControl(options=['position', 'transparent…

In [7]:
def calculate_cloudscore_custom(image):
    def rescale(img, exp, thresholds):
        """Helper function to rescale values linearly"""
        return (img.expression(exp, {'img': img})
                .subtract(thresholds[0])
                .divide(thresholds[1] - thresholds[0]))
    
    # Start with a score of 1.0 (clear)
    score = ee.Image(1.0)
    
    score = score.min(rescale(image, 'img.BLUE', [0.1, 0.3]))
    
    # 2. Clouds are bright in all visible bands
    score = score.min(rescale(image, 'img.RED + img.GREEN + img.BLUE', [0.2, 0.8]))
    
    # 3. Clouds are bright in all infrared bands
    score = score.min(rescale(image, 'img.NIR + img.SWIR1 + img.SWIR2', [0.3, 0.8]))
    return score
def mask_landsat_cloudscore_custom(image, cloud_threshold=0.8):
        # Calculate cloud score
    cloud_score = calculate_cloudscore_custom(image)
    
    # Invert score so 1 = clear, 0 = cloudy
    cloud_score = ee.Image(1).subtract(cloud_score).rename(['cloudscore'])
    
    # Create mask: keep pixels with score >= threshold
    cloud_mask = cloud_score.gte(cloud_threshold)
    
    # Apply mask and add cloudscore as a band
    return (image.updateMask(cloud_mask)
            .addBands(cloud_score)
            .copyProperties(image, image.propertyNames()))
def mask_landsat_sr(image,cloud_conf_thresh=2, shadow_conf_thresh=2, cirrus_conf_thresh=2):

        qa = image.select('QA_PIXEL')
            #Deterministic bits
            #fyi, (bit 3 is set to 1) and so on
        cloud_bit = 1 << 3
        shadow_bit = 1 << 4
        cirrus_bit = 1 << 2
        cloud_mask = qa.bitwiseAnd(cloud_bit).eq(0)
        shadow_mask = qa.bitwiseAnd(shadow_bit).eq(0)
        cirrus_mask = qa.bitwiseAnd(cirrus_bit).eq(0)
            #Confidence bits ---
        cloud_conf = qa.rightShift(8).bitwiseAnd(3)     # Bits 8–9
        shadow_conf = qa.rightShift(10).bitwiseAnd(3)   # Bits 10–11
        cirrus_conf = qa.rightShift(14).bitwiseAnd(3)   # Bits 14–15
            #Keep pixels below thresholds
        conf_mask = (cloud_conf.lt(cloud_conf_thresh)
                        .And(shadow_conf.lt(shadow_conf_thresh))
                        .And(cirrus_conf.lt(cirrus_conf_thresh)))
            #Final mask
        final_mask = cloud_mask.And(shadow_mask).And(cirrus_mask).And(conf_mask)
        return image.updateMask(final_mask).copyProperties(image, image.propertyNames())

In [8]:
masked = rename_collection.map(mask_landsat_cloudscore_custom)
Map.addLayer(masked.median().clip(aoi), l8_sr_visparam, 'L8 SR Custom CloudScore Masked Collection')
Map.addLayer(masked, l8_sr_visparam, 'L8 Collection masked custom')
Map

NameError: name 'rename_collection' is not defined

In [ ]:
qa_masked = dataset.map(
    lambda img: mask_landsat_sr(img, cloud_conf_thresh=3, shadow_conf_thresh=2, cirrus_conf_thresh=2))
m = geemap.Map()
m.addLayer(dataset,{'min': 0,'max': 0.4,'gamma': [0.95, 1.1, 1],'bands':['SR_B5', 'SR_B4', 'SR_B3']}, 'Initial Collection')
median_initial = dataset.median().clip(aoi)
m.addLayer(median_initial,{'min': 0,'max': 0.4,'gamma': [0.95, 1.1, 1],'bands':['SR_B5', 'SR_B4', 'SR_B3']}, 'Initial Median' )


m.addLayer(masked, l8_sr_visparam, 'Custom Mask')
median_masked = masked.median().clip(aoi)
m.addLayer(median_masked, l8_sr_visparam, 'custom mask median')


m.addLayer(qa_masked, {'min': 0,'max': 0.4,'gamma': [0.95, 1.1, 1],'bands':['SR_B5', 'SR_B4', 'SR_B3']}, 'QA Masked')
median_qa = qa_masked.median().clip(aoi)
m.addLayer(median_qa,  {'min': 0,'max': 0.4,'gamma': [0.95, 1.1, 1],'bands':['SR_B5', 'SR_B4', 'SR_B3']}, 'QA masked median')

m.centerObject(aoi, 7)
m

## Covariates/Predictor 

In [13]:
#start with spectral indices calculation
from epistemx.module_5 import spectral_transformation_calcultator, indexcategory, terrain_calculator, distance_calculator
spectral_transform = spectral_transformation_calcultator()
spectral_transform.list_category()
spectral_transform.indices_list(None)

2025-12-03 10:45:04,931 - root - INFO - Spectral transformation category:
2025-12-03 10:45:04,933 - root - INFO -   - vegetation: 9 indices
2025-12-03 10:45:04,934 - root - INFO -   - moisture: 2 indices
2025-12-03 10:45:04,934 - root - INFO -   - burn: 3 indices
2025-12-03 10:45:04,935 - root - INFO -   - water: 2 indices
2025-12-03 10:45:04,936 - root - INFO -   - soil_buildup: 3 indices
2025-12-03 10:45:04,936 - root - INFO -   - tascap: 1 indices
2025-12-03 10:45:04,937 - root - INFO - Supported spectral transformation:
2025-12-03 10:45:04,938 - root - INFO -  - NDVI: Normalized Difference Vegetation Index(Reference: https://doi.org/10.1016/0034-4257(79)90013-0)
2025-12-03 10:45:04,939 - root - INFO -  - GNDVI: Green Normalized Difference Vegetation Index(Reference: https://doi.org/10.1016/S0034-4257(96)00072-7 )
2025-12-03 10:45:04,939 - root - INFO -  - MSAVI: Modified Soil Adjusted Vegetation Index(Reference: https://doi.org/10.1016/0034-4257(94)90134-1)
2025-12-03 10:45:04,940 

In [ ]:
veg_cov = landsat_data.map(
    lambda img: spectral_transform.calculate_index(img, categories=[indexcategory.vegetation]))

median_veg_cov = final_data.get_temporal_composite(collection=veg_cov, aoi=aoi, reducer='median')

2025-12-03 10:49:00,888 - root - INFO - Starting spectral indices calculation
2025-12-03 10:49:00,889 - root - INFO - Calculating 9 indices: ['NDVI', 'GNDVI', 'MSAVI', 'OSAVI', 'ARVI', 'CVI', 'EVI', 'GBNDVI', 'MTVI1']
2025-12-03 10:49:00,890 - root - INFO - Successfully calculated NDVI: Normalized Difference Vegetation Index
2025-12-03 10:49:00,891 - root - INFO - Successfully calculated GNDVI: Green Normalized Difference Vegetation Index
2025-12-03 10:49:00,892 - root - INFO - Successfully calculated MSAVI: Modified Soil Adjusted Vegetation Index
2025-12-03 10:49:00,894 - root - INFO - Successfully calculated OSAVI: Optimized Soil Adjusted Vegetation Index
2025-12-03 10:49:00,895 - root - INFO - Successfully calculated ARVI: Atmospherically Resistant Vegetation Index
2025-12-03 10:49:00,896 - root - INFO - Successfully calculated CVI: Chlorophyll Vegetation Index
2025-12-03 10:49:00,898 - root - INFO - Successfully calculated EVI: Enhanced Vegetation Index
2025-12-03 10:49:00,899 - ro

EEException: Date: Parameter 'value' is required and may not be null.